# US Mortgage Portfolio Analysis — SQL

## Project Overview

This project analyzes a dataset of 50,000 US mortgage loans using SQL and SQLite in Python.

## Objectives

- Analyze mortgage loan distribution by purpose, occupancy, and state.
- Examine borrower credit scores, interest rates, LTV, and DTI.
- Analyze mortgage insurance percentage categories.
- Assess data completeness by checking missing values in key columns.

## Tools Used

- Python
- Pandas
- SQLite
- Jupyter Notebook

## Dataset

The analysis uses the cleaned dataset `US_Mortgage_Cleaned.csv`, containing 50,000 mortgage loan records.

## Business Goal

Identify portfolio patterns, borrower risk indicators, geographic concentration, and data-quality issues to support mortgage portfolio analysis.

In [1]:
import pandas as pd
import sqlite3


In [2]:
df=pd.read_csv("US_Mortgage_Cleaned.csv")

In [3]:
conn=sqlite3.connect("US_Mortgage.db")

In [4]:
df.to_sql("mortgage_loans",conn,if_exists="replace",index=False)

50000

In [5]:
query="""
PRAGMA table_info(mortgage_loans);
"""
pd.read_sql_query(query,conn)


,cid,name,type,notnull,dflt_value,pk
0,0,Classic FICO®,REAL,0,None,0
1,1,First Payment Date,INTEGER,0,None,0
2,2,First Time Homebuyer Indicator,TEXT,0,None,0
3,3,Maturity Date,INTEGER,0,None,0
4,4,Metropolitan Statistical Area (MSA) Or Metropo...,REAL,0,None,0
5,5,Mortgage Insurance Percentage (MI %),INTEGER,0,None,0
6,6,Number of Units,INTEGER,0,None,0
7,7,Occupancy Status,TEXT,0,None,0
8,8,Original Combined Loan-to-Value (CLTV),INTEGER,0,None,0
9,9,Original Debt-to-Income (DTI) Ratio,REAL,0,None,0


## Portfolio Overview

### Query 1: Total Number of Mortgage Loans

**Business Question:** How many mortgage loans are included in the dataset?

**Purpose:** Establish the total portfolio size and validate that the dataset contains the expected number of records.

In [6]:
# Total Number of Loans
query=""" 
Select Count(*)AS total_loans 
From mortgage_loans;
"""
pd.read_sql_query(query,conn)

,total_loans
0,50000


### Query 2: Loan Distribution by Purpose

**Business Question:** How are mortgage loans distributed across the different loan-purpose categories?

**Purpose:** Identify the most common loan-purpose category and compare the volume of loans across categories.

In [7]:
# Loan Distribution by Purpose
query="""
Select
  "Loan Purpose",
  Count(*)AS Total_loans
From mortgage_loans
Group BY "Loan Purpose"
Order BY total_loans Desc;
"""

pd.read_sql_query(query,conn)

,Loan Purpose,Total_loans
0,P,39627
1,C,6269
2,N,4104


### Query 3: Average Original UPB by Loan Purpose

**Business Question:** How does the average original unpaid principal balance (UPB) vary across loan-purpose categories?

**Purpose:** Compare the average original loan balance across categories and identify which category has the highest average loan amount.

In [8]:
# Average Original UPB by Loan Purpose
query="""
Select
  "Loan Purpose",
   Count(*)AS Total_loans,
   Round(AVG("Original UPB"), 2)As avg_original_upb
From mortgage_loans
Group BY "Loan Purpose"
Order BY avg_original_upb Desc;
"""
pd.read_sql_query(query,conn)

,Loan Purpose,Total_loans,avg_original_upb
0,N,4104,369332.12
1,P,39627,353013.73
2,C,6269,241756.10


### Query 4: Loan Distribution by Occupancy Status

**Business Question:** How are mortgage loans distributed across different property occupancy categories?

**Purpose:** Identify the most common occupancy category and understand the composition of the mortgage portfolio by occupancy status.

In [9]:
# Loan Distribution by Occupancy Status
query="""
Select
  "Occupancy Status",
   Count(*)AS total_loans
From mortgage_loans
Group By "Occupancy Status"
Order By total_loans Desc;
"""
pd.read_sql_query(query,conn)

,Occupancy Status,total_loans
0,P,45612
1,I,3437
2,S,951


## Borrower and Risk Analysis

### Query 5: Average FICO Score by Loan Purpose

**Business Question:** How does the average borrower FICO score vary across loan-purpose categories?

**Purpose:** Compare average credit scores across loan-purpose categories to understand differences in borrower credit profiles.

In [10]:
# Average FICO Score by Loan Purpose
query="""
Select
  "Loan Purpose",
   Count(*)AS Total_loans,
   Round(AVG("Classic FICO®"),2)As avg_fico
From mortgage_loans
Group By "Loan Purpose"
Order By avg_fico Desc;
"""
pd.read_sql_query(query,conn)

,Loan Purpose,Total_loans,avg_fico
0,P,39627,755.71
1,N,4104,754.96
2,C,6269,731.24


### Query 6: Average Interest Rate by Loan Purpose

**Business Question:** How does the average mortgage interest rate vary across loan-purpose categories?

**Purpose:** Compare average interest rates across categories and identify which category has the highest and lowest average rates.

In [11]:
# Average Interest Rate by Loan Purpose
query="""
Select
  "Loan Purpose",
   Count(*)AS Total_loans,
   Round(AVG("Original Interest Rate"),2) As avg_interest_rate
From mortgage_loans
Group By "Loan Purpose"
Order By avg_interest_rate Desc;
"""
pd.read_sql_query(query,conn)

,Loan Purpose,Total_loans,avg_interest_rate
0,C,6269,6.84
1,P,39627,6.70
2,N,4104,6.34


### Query 7: Loan Distribution by LTV Band

**Business Question:** How are mortgage loans distributed across different combined loan-to-value (CLTV) bands?

**Purpose:** Segment loans by CLTV ranges to understand the portfolio's distribution across different levels of borrower equity relative to property value.

In [12]:
# Loan Distribution by LTV Band
query="""
Select
  Case
     WHEN "Original Combined Loan-to-Value (CLTV)" IS NULL
        Then 'Missing CLTV'
     When "Original Combined Loan-to-Value (CLTV)"<=60
        Then 'Low LTV(<=60%)'
     When "Original Combined Loan-to-Value (CLTV)"<=80
        Then 'Moderate LTV(61-80%)'
     When "Original Combined Loan-to-Value (CLTV)"<=90
        Then 'High LTV(81-90%)'
     Else 'Very High LTV (>90%)'
  End As ltv_band,
  Count(*)AS Total_loans
From mortgage_loans
Group BY ltv_band
Order BY total_loans Desc;
"""
pd.read_sql_query(query,conn)

,ltv_band,Total_loans
0,Moderate LTV(61-80%),20555
1,Very High LTV (>90%),12139
2,Low LTV(<=60%),10993
3,High LTV(81-90%),6313


### Query 8: Loan Distribution by DTI Band

**Business Question:** How are mortgage loans distributed across different debt-to-income (DTI) ratio bands?

**Purpose:** Segment loans by DTI ranges to understand borrower debt burdens and identify the proportion of loans in higher DTI categories.

In [13]:
# Loan Distribution by DTI Band
query="""
Select
     Case 
       When "Original Debt-to-Income (DTI) Ratio" is Null
          Then 'Missing DTI'
       When "Original Debt-to-Income (DTI) Ratio"<=20
          Then 'Low DTI(<=20%)'
       When "Original Debt-to-Income (DTI) Ratio" <=35
          Then 'Moderate DTI(21-35%)'
       When "Original Debt-to-Income (DTI) Ratio" <=45
          Then 'High DTI(36-45%)'
       Else 'Very High DTI(>45%)'
     End As DTI_band,
     Count(*)AS Total_loans
From mortgage_loans
Group BY DTI_band
Order BY total_loans Desc;
"""
pd.read_sql_query(query,conn)

,DTI_band,Total_loans
0,High DTI(36-45%),19618
1,Moderate DTI(21-35%),15394
2,Very High DTI(>45%),12411
3,Low DTI(<=20%),2576
4,Missing DTI,1


### Query 9: Average Interest Rate by FICO Band

**Business Question:** How does the average mortgage interest rate vary across different borrower FICO score bands?

**Purpose:** Compare average interest rates across credit score categories to examine the relationship between borrower credit scores and mortgage pricing.

In [14]:
# Average Interest Rate by FICO Band
query="""
Select
  Case 
    When "Classic FICO®" is Null Then 'Missing Fico'
    When "Classic FICO®"< 650 Then 'Below 650'
    When "Classic FICO®"< 700 Then '650-699'
    When "Classic FICO®"< 750 Then '700-749'
    When "Classic FICO®"< 800 Then '750-799'
    Else '800+'
  End As fico_band,
  Count(*)AS Total_loans,
  Round(AVG("Original Interest Rate"),2)As avg_interest_rate
From mortgage_loans
Group BY fico_band
Order By CASE fico_band
    WHEN 'Below 650' THEN 1
    WHEN '650-699' THEN 2
    WHEN '700-749' THEN 3
    WHEN '750-799' THEN 4
    WHEN '800+' THEN 5
    WHEN 'Missing Fico' THEN 6
    ELSE 7
  END;
"""
pd.read_sql_query(query,conn)

,fico_band,Total_loans,avg_interest_rate
0,Below 650,1358,6.95
1,650-699,5872,6.86
2,700-749,12847,6.76
3,750-799,23461,6.63
4,800+,6446,6.58
5,Missing Fico,16,6.88


## Mortgage Insurance and Geography

### Query 10: Mortgage Insurance Distribution

**Business Question:** How are mortgage loans distributed across different mortgage insurance (MI) percentage categories?

**Purpose:** Analyze the frequency and portfolio share of each MI percentage category to understand the distribution of mortgage insurance values across the loan portfolio.

In [15]:
# Mortgage Insurance Distribution
query="""
Select
   "Mortgage Insurance Percentage (MI %)",
   Count(*)As total_loans,
   Round (
       Count(*)*100.0/(Select Count(*) From mortgage_loans),2)As portfolio_share_pct
   From mortgage_loans
   Group By "Mortgage Insurance Percentage (MI %)"
   Order By total_loans Desc;
   """
pd.read_sql_query(query,conn)
    
       


,Mortgage Insurance Percentage (MI %),total_loans,portfolio_share_pct
0,0,31834,63.67
1,30,7438,14.88
2,25,6775,13.55
3,12,1939,3.88
4,35,1396,2.79
5,18,309,0.62
6,16,235,0.47
7,6,73,0.15
8,15,1,0.00


### Query 11: Top 10 States by Loan Volume

**Business Question:** Which 10 U.S. states have the highest number of mortgage loans in the dataset?

**Purpose:** Identify the states with the largest loan volumes and understand the geographic concentration of the mortgage portfolio.

In [16]:
# Top 10 States by Loan Volume
query="""
 Select
    "Property State" AS property_state,
    Count(*)AS total_loans,
    Round(Count(*)*100.0/(Select Count(*)From mortgage_loans),2)As portfolio_share_pct
    From mortgage_loans
    Group BY "Property State"
    Order BY total_loans Desc LIMIT 10;
    """
pd.read_sql_query(query,conn)

,property_state,total_loans,portfolio_share_pct
0,TX,4619,9.24
1,FL,3935,7.87
2,CA,3647,7.29
3,IL,2431,4.86
4,OH,2225,4.45
5,PA,1895,3.79
6,MI,1893,3.79
7,NC,1759,3.52
8,GA,1727,3.45
9,NY,1668,3.34


## Data Quality

### Query 12: Missing Values and Data Quality Check

**Business Question:** Are there any missing values in key mortgage loan fields, such as FICO score, original UPB, interest rate, CLTV, and DTI?

**Purpose:** Assess the completeness of important loan and borrower attributes to identify potential data quality issues that could affect portfolio analysis and reporting.

In [17]:
# Missing-Value Analysis
query = """
SELECT
    COUNT(*) AS total_loans,
    SUM(CASE WHEN "Classic FICO®" IS NULL THEN 1 ELSE 0 END) AS missing_fico,
    SUM(CASE WHEN "Original UPB" IS NULL THEN 1 ELSE 0 END) AS missing_upb,
    SUM(CASE WHEN "Original Interest Rate" IS NULL THEN 1 ELSE 0 END) AS missing_interest_rate,
    SUM(CASE WHEN "Original Combined Loan-to-Value (CLTV)" IS NULL THEN 1 ELSE 0 END) AS missing_cltv,
    SUM(CASE WHEN "Original Debt-to-Income (DTI) Ratio" IS NULL THEN 1 ELSE 0 END) AS missing_dti
FROM mortgage_loans;
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 total_loans  missing_fico  missing_upb  missing_interest_rate  missing_cltv  missing_dti
       50000            16            0                      0             0            1


# Key Business Insights

1. **Portfolio Composition:** The dataset contains 50,000 mortgage loans. Loan-purpose category P represents 79.25% of the portfolio, while occupancy category P accounts for 91.22% of loans.

2. **Credit Profile:** Average FICO scores vary by loan purpose, ranging from 731.24 for category C to 755.71 for category P.

3. **Interest Rates:** Average interest rates decrease from 6.95% for borrowers with FICO scores below 650 to 6.58% for scores of 800 and above.

4. **CLTV Analysis:** 24.28% of loans have a CLTV above 90%, representing a segment worth further investigation for borrower equity and collateral risk.

5. **DTI Analysis:** 39.24% of loans fall within the 36–45% DTI band, while 24.82% have DTI above 45%.

6. **Geographic Concentration:** Texas, Florida, and California account for 24.40% of all loans. The top 10 states account for approximately 51.60%.

7. **Data Quality:** Only 16 FICO values and one DTI value are missing. Original UPB, interest rate, and CLTV have no missing values.

## Conclusion

The analysis highlights portfolio concentration, differences in borrower credit profiles and interest rates, and segments with higher CLTV and DTI values. Geographic analysis identifies the states with the largest loan volumes, while the data quality check confirms that the selected key fields are largely complete.

*Note: Loan-purpose, occupancy, and mortgage insurance codes should be interpreted using the dataset documentation.*